In [1]:
import pandas as pd
import os
import cv2
import numpy as np
import json
from scipy.signal import find_peaks

In [2]:
#Test Code| Version 2#
Year='1938'
Showa='13'
#Read Motherdata
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\"+Year+"\\DataFrame_Unit.json"
with open(path, 'r') as j:
     dta = json.loads(j.read())

In [3]:
#Test code| Version 2#
df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'_Unit.csv')
df=df.drop(df.columns[0], axis=1)

PositionList=['Manager','Leader','Admin','Outsource','Engineer1','Engineer2','Clerk1','Clerk2','Worker','TempWorker','Inspector','Guard','Driver','Nurse','Medic','Police','Pharma','Monitor']

In [4]:
### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

def Clova(Dept,Office,Unit,ImagePage,LeftEdge,RightEdge):
    api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
    secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Unit+"\\"
    stream = open(path+"Image"+str(ImagePage)+'.png', "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    HH,WW=img.shape[:2]
    cropped = img[0:100,LeftEdge:RightEdge]
    
    try:    
        temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
        cv2.imwrite(temp_path+"Temp.jpg",cropped)
        with open(temp_path+"Temp.jpg",'rb') as f:
            file_data = f.read()

        request_json = {
                'images': [
                    {
                        'format': 'jpg',
                        'name': 'demo',
                        'data':base64.b64encode(file_data).decode()}],
                'requestId': str(uuid.uuid4()),
                'version': 'V2',
                'timestamp': int(round(time.time() * 1000)),
                'lang':'ja'
                }
        payload = json.dumps(request_json).encode("UTF-8")
        headers = {'X-OCR-SECRET': secret_key,
                  'Content-Type': 'application/json'}
        response = requests.request("POST", api_url, headers=headers, data = payload)
        Json=json.loads(response.text)['images'][0]
        if 'fields' not in list(Json.keys()):
            return 'NA'
        if Json['fields']==[]:
            return 'NA'
        if Json['fields']!=[]:
            Text=''.join([d['inferText'] for d in Json['fields']])
        return Text
    except:
        return ([])


In [5]:
# Function for returning x coordinate of Position name
def Detect_Position(Frame,Position):
    ### Get Coordinates ###
    try:
        if Position=='Manager': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('主事|主|毒|寿|畫')])
            return(Edge)

        if Position=='Leader': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('課長|課')])
            return(Edge)

        if Position=='Engineer1': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('技師|師|技術|支')])
            return(Edge)

        if Position=='Engineer2': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('技手|手|个')])
            return(Edge)

        if Position=='Admin': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('事務員')])
            return(Edge)

        if Position=='Outsource': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('嘱託員|嘱託|嘱|託|呪')])
            return(Edge)

        if Position=='Clerk1': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('皆|書記|言')])
            return(Edge)

        if Position=='Clerk2': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('収納|收納')])
            return(Edge)

        if Position=='Clerk3': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('社會書')])
            return(Edge)

        if Position=='Worker': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('雇|届|扉|履|雁|展|扁|羅|風|厨')])
            return(Edge)

        if Position=='TempWorker': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('臨時')])
            return(Edge)

        if Position=='Inspector': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('視学|視學')])
            return(Edge)
        
        if Position=='Guard': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('守衛長')])
            return(Edge)
        
        if Position=='Driver': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('運轉|轉')])
            return(Edge)
        
        if Position=='Nurse': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('保健婦')])
            return(Edge)

        if Position=='Medic': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('醫員')])
            return(Edge)

        if Position=='Police': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('警員')])
            return(Edge)

        if Position=='Pharma': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('調薬員|藥員')])
            return(Edge)

        if Position=='Monitor': 
            Edge=int(Frame['Second'][Frame['Text'].str.contains('監視')])
            return(Edge)
    except:
        return 'NA'

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)


In [6]:
def Clova2(Dept,Office,Unit,ImagePage):
    api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/2038/42ead006bc8f828b6079063216fa47aa99c9761923e949392e11eec8784c8db2/infer'
    secret_key = 'bUdOUG5CR3FJb0VqRmZrSE9EWUtIem5LcFllRGxiUnA='

    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Unit+"\\"
    stream = open(path+"Image"+str(ImagePage)+'.png', "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    HH,WW=img.shape[:2]
    cropped = img[0:HH,0:WW]
    
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
    cv2.imwrite(temp_path+"Temp.jpg",cropped)
    with open(temp_path+"Temp.jpg",'rb') as f:
        file_data = f.read()
    
    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)
    
    return Json    



In [7]:
def Organize(Dict,EndPage):
    ##Add Ending information
    Frame=pd.DataFrame.from_dict(Dict,orient='index')
    Frame=Frame.sort_values(by = ['Starting_Page', 'XLocation'], ascending = [True, False])
    Frame["EndLocation"]=list(Frame['XLocation'].shift(-1))[:-1]+list([0])
    Frame["EndPage"]=list(Frame['Starting_Page'].shift(-1))[:-1]+list([EndPage])
    Frame=Frame.to_dict('index')
    return(Frame)

In [8]:
def Show(Position,Page):
    PosiList=list(Dict.keys())
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+'\\'+Dept+"\\"+Office+"\\"+Unit+"\\"+Position+"\\Image"+str(Page)+".png"
    stream = open(path, "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    HH=img.shape[:2][0]
    for Posi in PosiList:
        XCord=Dict[Posi]['XLocation']
        cv2.line(img, (XCord,0), (XCord,HH), (0,0,0), 3)
    cv2.imshow(Office,img)
    cv2.waitKey(0)

In [9]:
def Find_cell(Dept,Office,Unit,ImagePage):
    path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\'
    path=os.path.join(path,Year,Dept,Office,Unit)
    stream = open(path+"\\Image"+str(ImagePage)+'.png', "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

    HH,WW=img.shape[:2]
    Top=150
    while Top>50:        
        cropped=img[0:Top, 0:WW]

        # convert to grayscale
        gray = cv2.cvtColor(cropped,cv2.COLOR_BGR2GRAY)
        # threshold gray image
        thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)[1]

        # count number of non-zero pixels in each column and row. 
        countCol = np.count_nonzero(thresh, axis=0)

        ### This finds the height of the lowest valley
        peaks, _ = find_peaks(countCol, distance=15)

        Frame={'First':peaks,'Second':np.roll(peaks,-1).tolist()[:-1]+list([WW])}
        Frame=pd.DataFrame.from_dict(Frame,orient='index').transpose()
        ColNum=len(Frame)
        Frame['Text']=list(map(Clova,[Dept]*ColNum,[Office]*ColNum,[Unit]*ColNum,[ImagePage]*ColNum,Frame['First'],Frame['Second']))

        #Find Starting X coordinates of Positions.
        #First Loop: 
        CordList=list(map(Detect_Position,[Frame]*len(PositionListNow),PositionListNow))
        Positions= {PositionListNow[i]:CordList[i] for i in range(len(PositionListNow))}
        Positions = {k:v for k,v in Positions.items() if v}    

        PosiList = list(Positions.keys())
        CordList = list(Positions.values())

        CordDictNow= {PosiList[i]: {'XLocation':CordList[i],'Starting_Page':ImagePage} for i in range(len(PosiList))}
        CordDictNow = {k:v for k,v in CordDictNow.items() if v['XLocation']!='NA'}
        if len(CordDictNow)<=len(CordDict):
            break
        else:
            print('updating...'+str(len(CordDict))+' to '+str(len(CordDictNow)))
            display((CordDictNow.keys()))
            CordDict.update(CordDictNow)
            Top=Top-20
    print('Finished first attempt at scanning...')
    print('Result of Initial Scan was...')
    display(CordDict)

    CordDictNow['First']={'XLocation':HH}
    Frame=pd.DataFrame.from_dict(CordDictNow,orient='index')
    Frame=Frame.sort_values(by=['XLocation'],ascending=False)
    Frame['EndLocation']=Frame['XLocation'].shift(-1)
    Frame.loc[Frame.index[-1], 'EndLocation'] = 0
    Frame["EndLocation"].apply(np.int64)
    
    print('Detected Positions were...')
    display(Frame)

    RowNum=len(Frame)
    print('Refining Scans...')
    
    CordDict2={}
    for i in range(0,RowNum):    
        Row=Frame.iloc[i]
        Righ=int(Row['XLocation'])
        Left=int(Row['EndLocation'])
        if abs(Righ-Left)<5:
            continue

        cropped=img[0:Top+30,Left:Righ]
        HH,WW=cropped.shape[:2]
        
        # convert to grayscale
        gray = cv2.cvtColor(cropped,cv2.COLOR_BGR2GRAY)
        # threshold gray image
        thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)[1]

        # count number of non-zero pixels in each column and row. 
        countCol = np.count_nonzero(thresh, axis=0)

        ### This finds the height of the lowest valley
        peaks, _ = find_peaks(countCol, distance=15)
        peaks = peaks+Left

        Frame2={'First':peaks,'Second':np.roll(peaks,-1).tolist()[:-1]+list([WW+Left])}
        Frame2=pd.DataFrame.from_dict(Frame2,orient='index').transpose()
        if len(Frame2)<=1:
            continue
        
        ColNum=len(Frame2)
        Frame2['Text']=list(map(Clova,[Dept]*ColNum,[Office]*ColNum,[Unit]*ColNum,[ImagePage]*ColNum,Frame2['First'],Frame2['Second']))
        
        #Find Starting X coordinates of Positions.
        #First Loop: 
        CordList=list(map(Detect_Position,[Frame2]*len(PositionListNow),PositionListNow))
        Positions= {PositionListNow[i]:CordList[i] for i in range(len(PositionListNow))}
        Positions = {k:v for k,v in Positions.items() if v}   

        PosiList = list(Positions.keys())
        CordList = list(Positions.values())

        CordDictNow2= {PosiList[i]: {'XLocation':CordList[i],'Starting_Page':ImagePage} for i in range(len(PosiList))}
        CordDictNow2 = {k:v for k,v in CordDictNow2.items() if v['XLocation']!='NA'}
        print('Scanned '+str(i+1)+'th crop...')
        print('Results is...')
        display(CordDictNow2)
        #cv2.imshow(str(i)+'th Image',cropped)
        #cv2.waitKey(0)

        CordDict2.update(CordDictNow2)
    CordDict.update(CordDict2)
    return(CordDict)

In [10]:
def hconcat_resize_min(im_list, interpolation=cv2.INTER_CUBIC):
    h_min = min(im.shape[0] for im in im_list)
    im_list_resize = [cv2.resize(im, (int(im.shape[1] * h_min / im.shape[0]), h_min), interpolation=interpolation)
                      for im in im_list]
    return cv2.hconcat(im_list_resize)

def Check(df,dta,K):
    for n in range(0,K):
        #Extract key info of office
        Row  = df.iloc[n]

        ###Collect Location information###
        Dept=Row["Dept"]
        Office=Row["Office"]
        Unit=Row["Unit"]
        
        print('['+str(n)+',"'+Dept+'","'+Office+'","'+Unit+'"]')      
        StrPage=1
        try:
            EndPage=dta[Year][Dept][Office]['Units'][Unit]['EndPage']-dta[Year][Dept][Office]['Units'][Unit]['Page']+1

            path=os.path.join('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level',Year,Dept,Office,Unit)
            stream = open(path+"\\Image"+str(StrPage)+'.png', "rb")
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            HH,WW=img.shape[:2]
            oldimg=img.copy()[0:200,0:WW]

            for Page in range(StrPage+1,EndPage+1):
                path=os.path.join('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level',Year,Dept,Office,Unit)
                stream = open(path+"\\Image"+str(Page)+'.png', "rb")
                bytes = bytearray(stream.read())
                numpyarray = np.asarray(bytes, dtype=np.uint8)
                newimg = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
                HH,WW=newimg.shape[:2]
                newimg = newimg[0:200,0:WW]
                oldimg= hconcat_resize_min((newimg,oldimg))
            oldimg=cv2.copyMakeBorder(oldimg,20,20,20,20,cv2.BORDER_CONSTANT)

            PosiList=list(dta[Year][Dept][Office]['Units'][Unit]['Positions'].keys())
            for Posi in PosiList:
                StrLoc=int(dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['XLocation'])
                StrPage=int(dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['Starting_Page'])

                path=os.path.join('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level',Year,Dept,Office,Unit)
                stream = open(path+"\\Image"+str(StrPage)+'.png', "rb")
                bytes = bytearray(stream.read())
                numpyarray = np.asarray(bytes, dtype=np.uint8)
                newimg = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
                HH,WW=newimg.shape[:2]
                Annotate=newimg.copy()[0:200,0:WW]
                cv2.line(Annotate,(StrLoc,0),(StrLoc,HH),(255,0,0),2)
                Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
                oldimg=hconcat_resize_min((Annotate,oldimg))

            oldimg=cv2.resize(oldimg, None, fx = 0.60, fy = 0.60)
            cv2.imshow(Dept+Office,oldimg)
            cv2.waitKey(0)
        except:
            continue
    cv2.destroyAllWindows()
    cv2.waitKey(0)


In [ ]:
FailedList=[]
for n in range(0,len(df)+1):
    PositionListNow=PositionList.copy()
    #Extract key info of office
    Row  = df.iloc[n]

    ###Collect Location information###
    Dept=Row["Dept"]
    Office=Row["Office"]
    Unit=Row['Unit']
    print('['+str(n)+',"'+Dept+'","'+Office+'","'+Unit+'"]')

    StrPage=1
    EndPage=dta[Year][Dept][Office]['Units'][Unit]['EndPage']-dta[Year][Dept][Office]['Units'][Unit]['Page']+1
    Page_Range=range(StrPage,EndPage+1)

    CordDict={}
    for ImagePage in Page_Range:
        try:
            CordDict.update(Find_cell(Dept,Office,Unit,ImagePage))
        except:
            print('Failed at '+Dept+Office+Unit)
            FailedList.append((n,Dept,Office,Unit))
            continue
            
    if len(CordDict)==0:
        FailedList.append((n,Dept,Office,Unit))
        continue
    dta[Year][Dept][Office]['Units'][Unit]['Positions']=Organize(CordDict,EndPage)

[0,"Admin","人事課","庶務掛"]
updating...0 to 1


dict_keys(['Manager'])

Finished first attempt at scanning...
Result of Initial Scan was...


{'Manager': {'XLocation': 69, 'Starting_Page': 1}}

Detected Positions were...


...

{'Manager': {'XLocation': 204, 'Starting_Page': 1},
 'Clerk1': {'XLocation': 160, 'Starting_Page': 1},
 'Worker': {'XLocation': 46, 'Starting_Page': 1},
 'Guard': {'XLocation': 63, 'Starting_Page': 1}}

Detected Positions were...


,XLocation,Starting_Page,EndLocation
First,462,NaN,204.0
Manager,204,1.0,160.0
Clerk1,160,1.0,67.0
Guard,67,1.0,46.0
Worker,46,1.0,0.0


Refining Scans...
Scanned 2th crop...
Results is...


{}

In [ ]:
Check(df,dta,40)

In [21]:
dta[Year]['Admin']['人事課']['Units']["庶務掛"]

{'Page': 1,
 'StartLocation': 101,
 'EndPage': 2,
 'EndLocation': 402,
 'Page_Range': [1, 2],
 'Positions': {'Manager': {'XLocation': 69,
   'Starting_Page': 1,
   'EndLocation': 64.0,
   'EndPage': 2.0},
  'Worker': {'XLocation': 64,
   'Starting_Page': 2,
   'EndLocation': 31.0,
   'EndPage': 2.0},
  'TempWorker': {'XLocation': 31,
   'Starting_Page': 2,
   'EndLocation': 0.0,
   'EndPage': 2.0}}}

In [72]:
for n in range(0,50):
    #Extract key info of office
    Row  = df2.iloc[n]

    ###Collect Location information###
    Dept=Row["Dept"]
    Office=Row["Office"]
    Unit=Row['Unit']
    
    StrPage=1
    EndPage=int(dta[Year][Dept][Office]['Units'][Unit]['EndPage'])-int(dta[Year][Dept][Office]['Units'][Unit]['Page'])+1
    print('Page Range is '+str(StrPage)+str(EndPage))
    
    path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\'
    path=os.path.join(path,Year,Dept,Office,Unit)
    stream = open(path+"\\Image"+str(StrPage)+'.png', "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    oldimg=img.copy()
    
    for Page in range(StrPage+1,EndPage+1):
        stream = open(path+"\\Image"+str(Page)+'.png', "rb")
        bytes = bytearray(stream.read())
        numpyarray = np.asarray(bytes, dtype=np.uint8)
        newimg = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
        oldimg=np.concatenate((newimg,oldimg), axis=1)
    oldimg=cv2.copyMakeBorder(oldimg,20,20,20,20,cv2.BORDER_CONSTANT)
    
    PosiList=list(dta[Year][Dept][Office]['Units'][Unit]['Positions'].keys())
    for Posi in PosiList:        
        StrLoc=int(dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['XLocation'])
        Page=int(dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['Starting_Page'])

        path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\'
        path=os.path.join(path,Year,Dept,Office,Unit)
        stream = open(path+"\\Image"+str(Page)+'.png', "rb")
        bytes = bytearray(stream.read())
        numpyarray = np.asarray(bytes, dtype=np.uint8)
        img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
        HH,WW=img.shape[:2]

        Annotate=img.copy()
        cv2.line(Annotate,(StrLoc,0),(StrLoc,HH),(255,0,0),2)
        Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
        oldimg=np.concatenate((Annotate,oldimg), axis=1)
    cv2.imshow(Dept+Office+Unit+Posi,oldimg)
    cv2.waitKey(0)

Page Range is 12
Page Range is 12
Page Range is 12
Page Range is 11
Page Range is 11
Page Range is 11
Page Range is 11


IndexError: single positional indexer is out-of-bounds

In [75]:
df2

,Dept,Office,Unit,Year,Position
0,Admin,人事課,庶務掛,1938,Manager
1,Admin,人事課,庶務掛,1938,Worker
2,Admin,人事課,庶務掛,1938,TempWorker
3,Admin,人事課,人事掛,1938,Manager
4,Admin,人事課,人事掛,1938,Clerk1
5,Admin,人事課,給與掛,1938,Engineer2
6,Admin,人事課,給與掛,1938,Clerk1


In [12]:
json_object = json.dumps(dta, indent=4,
                        cls=NpEncoder)
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+str(Year)+'\\'
with open(save_path+"DataFrame_Position.json", "w") as outfile:
    outfile.write(json_object)

In [13]:
df2=pd.DataFrame(columns=['Dept','Office','Unit','Year','Position'])
for n in range(0,len(df)):
    Row=df.iloc[n]
    Office=Row['Office']
    Dept=Row['Dept']
    Unit=Row['Unit']
    try:
        PosiList=list(dta[Year][Dept][Office]['Units'][Unit]['Positions'].keys())
        for Posi in PosiList:
            NewRow={'Office':Office,'Dept':Dept,'Unit':Unit,'Year':Year,'Position':Posi}
            df2 = df2.append(NewRow, ignore_index = True)
    except:
        continue
df2.to_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'_Final.csv')

C:\Temp\ipykernel_25336\2579514942.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df2 = df2.append(NewRow, ignore_index = True)
C:\Temp\ipykernel_25336\2579514942.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df2 = df2.append(NewRow, ignore_index = True)
C:\Temp\ipykernel_25336\2579514942.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df2 = df2.append(NewRow, ignore_index = True)
C:\Temp\ipykernel_25336\2579514942.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df2 = df2.append(NewRow, ignore_index = True)
C:\Temp\ipykernel_25336\2579514942.py:11: FutureWarning: The frame.append method is deprecated and will be r

In [52]:
PositionListNow=PositionList.copy()
n=15
#Extract key info of office
Row  = df.iloc[n]
###Collect Location information###
Dept=Row["Dept"]
Office=Row["Office"]
Unit=Row['Unit']

StrPage=1
EndPage=dta[Year][Dept][Office]['Units'][Unit]['EndPage']-dta[Year][Dept][Office]['Units'][Unit]['Page']+1
Page_Range=range(StrPage,EndPage+1)

CordDict={}
for ImagePage in Page_Range:
    CordDictNow=Find_cell(Dept,Office,Unit,ImagePage)
    display(CordDictNow)
    CordDict.update(CordDictNow)
dta[Year][Dept][Office]['Units'][Unit]['Positions']=Organize(CordDict,EndPage)

updating...0 to 2


dict_keys(['Manager', 'Engineer1'])

Finished first attempt at scanning...
Result of Initial Scan was...


{'Manager': {'XLocation': 193, 'Starting_Page': 1},
 'Engineer1': {'XLocation': 152, 'Starting_Page': 1}}

Detected Positions were...


...

Detected Positions were...


,XLocation,EndLocation
First,468,0.0


Refining Scans...
Scanned 1th crop...
Results is...


{}

{'Manager': {'XLocation': 193, 'Starting_Page': 1},
 'Engineer1': {'XLocation': 152, 'Starting_Page': 1},
 'Outsource': {'XLocation': 259, 'Starting_Page': 2},
 'Clerk1': {'XLocation': 457, 'Starting_Page': 2},
 'Worker': {'XLocation': 423, 'Starting_Page': 2},
 'TempWorker': {'XLocation': 293, 'Starting_Page': 2}}

In [16]:
FailRate=len(FailedList)/len(df)
if len(FailedList)/len(df)<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList)/len(df))))
else:
    print('残念...Success Rate is '+str(1-(len(FailedList)/len(df))))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Position'] = 1-FailRate
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index=False )

Fantastic!! Success Rate is 0.9881656804733728


{'Admin': {'秘書課': {'Starting_Page': 2,
   'Office_X1': 480,
   'Ending_Page': 2,
   'Office_X2': 178,
   'Page_Range': [2]},
  '人事課': {'Starting_Page': 2,
   'Office_X1': 168,
   'Ending_Page': 4,
   'Office_X2': 377,
   'Page_Range': [2, 3, 4],
   'Units': {'庶務掛': {'Page': 1,
     'StartLocation': 101,
     'EndPage': 2,
     'EndLocation': 402,
     'Page_Range': [1, 2]},
    '人事掛': {'Page': 2,
     'StartLocation': 402,
     'EndPage': 2,
     'EndLocation': 267,
     'Page_Range': [2]},
    '給與掛': {'Page': 2,
     'StartLocation': 267,
     'EndPage': 2,
     'EndLocation': 132,
     'Page_Range': [2]},
    '福利掛': {'Page': 2,
     'StartLocation': 132,
     'EndPage': 3,
     'EndLocation': 0,
     'Page_Range': [2, 3]},
    'List': ['庶務掛', '人事掛', '給與掛', '福利掛']}},
  '文書課': {'Starting_Page': 4,
   'Office_X1': 367,
   'Ending_Page': 6,
   'Office_X2': 145,
   'Page_Range': [4, 5, 6],
   'Units': {'庶務掛': {'Page': 1,
     'StartLocation': 315,
     'EndPage': 1,
     'EndLocation': 20